In [13]:
#Baseline Algorithms in Flat Environment
from Multi_Agent_Environment_Flat import CustomEnvironmentFlat
from pettingzoo.test import parallel_api_test
import gymnasium as gym
env = CustomEnvironmentFlat()
parallel_api_test(env, num_cycles=1_000_000)


Passed Parallel API test


In [14]:
#Random Policy
import random
import numpy as np
# Initialize the env
episode_rewards=[]
# Use a fixed agent list
def RandomPolicy(env,input_seed):
    all_agents = ["satellite1", "satellite2", "satellite3"]
    agent_rewards = {agent: [] for agent in all_agents}
    # Track done flags
    obs, info = env.reset(seed=input_seed)
    terminated = {agent: False for agent in all_agents}
    truncated = {agent: False for agent in all_agents}
    total_rewards = {agent: 0 for agent in all_agents}
    while not all([terminated[a] or truncated[a] for a in all_agents]):
        actions = {
            agent: env.action_space(agent).sample()
            for agent in all_agents
            if not (terminated[agent] or truncated[agent])
        }
        #print("Agent Actions")
        #print(actions)
        observations, rewards, terminated, truncated, info = env.step(actions)
        for agent, r in rewards.items():
            total_rewards[agent] += r
        
  
    satellite1_info=info['satellite1']
    Overall_Delivered_Packets=satellite1_info['Delivered_Packets']
    Overall_Excess_Energy=satellite1_info['Total_Energy_Expended']
    Overall_Number_of_Contacts_Used=satellite1_info['Total_Number_of_Contacts_Used']
    num_of_collisions=satellite1_info['Number_of_Collisions']
    total_initial_data_volume=satellite1_info['Total_Initial_Satellite_Data_Volume']
    
    agent_specific_delivered_packets=[]
    agent_specific_excess_energy=[]
    agent_specific_num_of_contacts=[]
    agent_specific_initial_data_volume=[]
    for a in all_agents:
        sat_info=info[a]
        agent_specific_delivered_packets.append(sat_info['satellite_delivered_packets'])
        agent_specific_excess_energy.append(sat_info['satellite_energy_expended'])
        agent_specific_num_of_contacts.append(sat_info['satellite_number_of_contacts_used'])
        agent_specific_initial_data_volume.append(sat_info['satellite_initial_data_volume'])
        
    env.close()
    return total_rewards,Overall_Delivered_Packets,Overall_Excess_Energy,Overall_Number_of_Contacts_Used,agent_specific_delivered_packets,agent_specific_excess_energy,agent_specific_num_of_contacts,total_initial_data_volume,agent_specific_initial_data_volume


In [100]:
#Adaptive Sorting Heuristic
def AdaptiveSorting(env,seed):
    init_observations,init_info=env.reset()
    #Next need to conduct necessary calculations to obtain the strategy for the actions
    #With full observability we can just take satellite 1's observation space as we can see everything
    #that satellite 2 and 3 can see
    sat1_init_observations=init_observations["satellite1"]
    print(sat1_init_observations)
    #Need to reconstruct contact_plan
    first_150 = sat1_init_observations[:150]
    contact_plan = first_150.reshape(30, 5)
    print("Contact Plan")
    print(contact_plan)
    #Need to add order identifier to each contact
    column_id=np.zeros([30,1])
    for i in range(0,30):
        column_id[i]=i
    contact_plan= np.column_stack((contact_plan, column_id))
    #Need to manage all the satellite buffers and minimize collisions
    #Dictionary that contains the remaining packets to send\
    
    satellite_buffers=sat1_init_observations[150:153]
    print(satellite_buffers)
    #Next need to convert this to something easily handled in the heuristic algorithm
    all_agents = ["satellite1", "satellite2", "satellite3"]
    satellite_buffers_algo=[]
    for a in range(0,3):
        satellite_buffers_algo.append(satellite_buffers[a])
    
    
    
    #Next we need to sort the contact plan by cloud cover value
    print("Contact Plan before Sorting")
    print(contact_plan)
    #First we are going to extract the contacts that are not padded
    real_contact_plan = contact_plan[contact_plan[:, 0] != -1]
    print("Contact plan to be sorted")
    print(real_contact_plan)




    
    sorted_contact_plan=sorted(real_contact_plan,key=lambda row: row[0])
    print("Sorted Matrix")
    print(sorted_contact_plan)
    #Next we need to allocate the contacts based upon the buffers of the satellites
    estimated_capacity=0
    for i in sorted_contact_plan:
        if i[0]>= 0:
            estimated_capacity+=round(i[0]*10)
        
    print("Estimated Capacity")
    print(estimated_capacity)
    
    #Now we have our initial action strategy
    #We implement while loop here
    actions=[]#Options are 0,1,2,3
    
    for contact in sorted_contact_plan:
        print("Current Contact")
        print(contact)
        contact_action_options=[]
        if contact[0]<1:
            for l in range(0,3):
                contact_action_options.append(contact[2+l])
            print("Action Options")
            print(contact_action_options)
            #Next we need to check how much data is left on each satellite that can transmit
            maximum_data_remaining=0
            satellite_counter=0
            max_satellite_index=None
            for sat in satellite_buffers_algo:
                
                if sat>maximum_data_remaining:
                    max_satellite_index=satellite_counter
                    maximum_data_remaining=sat
                satellite_counter+=1
            #print("Max data remaining in a satellite buffer")
            #print(maximum_data_remaining)
            print("Satellite buffers")
            print(satellite_buffers_algo)
            
            #Next need to check if the maximum remaining data is still 0
            if maximum_data_remaining==0:
                actions.append([0,contact[5]])
                print("Actions")
                print(actions)
                #We don't want to send as no satellite that can see the ground station
                #has data to send
            else:
                #Next we will use the satellite that has the most data to send
                
                actions.append([max_satellite_index+1,contact[5]])
                #next we must update the estimated remaining data
                #Expected Number of packets sent
                satellite_buffers_algo[max_satellite_index]=satellite_buffers_algo[max_satellite_index]-round(contact[0]*contact[1])
                if satellite_buffers_algo[max_satellite_index]<0:
                    satellite_buffers_algo[max_satellite_index]=0
                
                
            
            
        else:
            actions.append([0,contact[5]])
    
    #Next we need to actually implement this algorithm
    #First we need to sort the actions properly
    sorted_actions=sorted(actions,key=lambda row:row[1])
    
    print("Initial Sorted Actions")
    print(sorted_actions)
    #Convert the sorted_actions to a dictionary format
    true_actions={agent: 0 for agent in all_agents}
    index_counter=1
    for agent in all_agents:
        
        if sorted_actions[0]==index_counter:
            true_actions[agent]=1
        index_counter+=1
        
    
    
    #Next we actions are actually implemented we check the environment and then conduct sorting and allocation
    #Again
    terminated = {agent: False for agent in all_agents}
    truncated = {agent: False for agent in all_agents}
    
    timestep=0
    total_rewards=0
    while not all([terminated[a] or truncated[a] for a in all_agents]):
        
        index_counter=1
        for agent in all_agents:
            if not (terminated[agent] or truncated[agent]):
                if sorted_actions[timestep][0]==index_counter:
                    true_actions[agent]=1
            index_counter+=1
        print("Agent Actions")
        print(true_actions)
        observations, rewards, terminated, truncated, info = env.step(true_actions)
        for a in all_agents:
            total_rewards+=rewards[a]
        #Check if any data was sent
        
        
        sat1_init_observations=observations["satellite1"]
        new_satellite_buffers=sat1_init_observations[150:153]
        buffer_change_flag=0
        for i in range(len(satellite_buffers)):
            if satellite_buffers[i]>new_satellite_buffers[i]:
                buffer_change_flag=1
        
        #If so we need to recalculate
        satellite_buffers=new_satellite_buffers
        if buffer_change_flag==1:
            first_150 = sat1_init_observations[:150]
            contact_plan = first_150.reshape(30, 5)
            print("Contact Plan")
            print(contact_plan)
            print(contact_plan)
            #Need to modify the contact plan to only include current observations
            updated_contact_plan=[]
            for i in range(timestep,len(contact_plan)):
                updated_contact_plan.append(contact_plan[i])
            print("Updated Contact Plan")
            print(updated_contact_plan)
            #Need to add order identifier to each contact
            column_id=np.zeros([30,1])
            for i in range(0,30):
                column_id[i]=i
            updated_contact_plan= np.column_stack((updated_contact_plan, column_id))
            #Need to manage all the satellite buffers and minimize collisions
            #Dictionary that contains the remaining packets to send
            #Next need to convert this to something easily handled in the heuristic algorithm
            all_agents = ["satellite1", "satellite2", "satellite3"]
            satellite_buffers_algo=[]
            print(satellite_buffers)
            for a in range(0,3):
                satellite_buffers_algo.append(satellite_buffers[a])
            
            
            
            #Next we need to sort the contact plan by cloud cover value
            sorted_contact_plan=sorted(updated_contact_plan,key=lambda row: row[0])
            print("Sorted Matrix")
            print(sorted_contact_plan)
            #Next we need to allocate the contacts based upon the buffers of the satellites
            estimated_capacity=0
            for i in sorted_contact_plan:
                if i[0]>= 0:
                    estimated_capacity=round(i[0]*10)
            
            print("Estimated Capacity")
            print(estimated_capacity)
            
            #Now we have our initial action strategy
            #We implement while loop here
            actions=[]#Options are 0,1,2,3
            for contact in sorted_contact_plan:
                contact_action_options=[]
                if contact[0]<1:
                    for l in range(0,3):
                        contact_action_options.append(contact[2+l])
                        #Next we need to check how much data is left on each satellite that can transmit
                        maximum_data_remaining=0
                        satellite_counter=0
                        max_satellite_index=None
                        for sat in satellite_buffers_algo:
                            if sat>maximum_data_remaining:
                                max_satellite_index=satellite_counter
                                maximum_data_remaining=sat
                            satellite_counter+=1
                        #Next need to check if the maximum remaining data is still 0
                        if maximum_data_remaining==0:
                            actions.append([0,contact[5]])
                            #We don't want to send as no satellite that can see the ground station
                            #has data to send
                        else:
                            #Next we will use the satellite that has the most data to send
                            actions.append([max_satellite_index+1,contact[5]])
                            #next we must update the estimated remaining data
                            #Expected Number of packets sent
                            satellite_buffers_algo[max_satellite_index]=satellite_buffers_algo[max_satellite_index]-round(contact[0]*contact[1])
                            if satellite_buffers_algo[max_satellite_index]<0:
                                satellite_buffers_algo[max_satellite_index]=0
                            
                            
                        
                        
                    else:
                        actions.append([0,contact[5]])
            
            #Next we need to actually implement this algorithm
            #First we need to sort the actions properly
            sorted_actions=sorted(actions,key=lambda row:row[1])
            
    
        timestep+=1
        print("Rewards")
        print(rewards)
        
    satellite1_info=info['satellite1']
    Overall_Delivered_Packets=satellite1_info['Delivered_Packets']
    Overall_Excess_Energy=satellite1_info['Total_Energy_Expended']
    Overall_Number_of_Contacts_Used=satellite1_info['Total_Number_of_Contacts_Used']
    num_of_collisions=satellite1_info['Number_of_Collisions']
    total_initial_data_volume=satellite1_info['Total_Initial_Satellite_Data_Volume']
    
    agent_specific_delivered_packets=[]
    agent_specific_excess_energy=[]
    agent_specific_num_of_contacts=[]
    agent_specific_initial_data_volume=[]
    for a in all_agents:
        sat_info=info[a]
        agent_specific_delivered_packets.append(sat_info['satellite_delivered_packets'])
        agent_specific_excess_energy.append(sat_info['satellite_energy_expended'])
        agent_specific_num_of_contacts.append(sat_info['satellite_number_of_contacts_used'])
        agent_specific_initial_data_volume.append(sat_info['satellite_initial_data_volume'])
        
    env.close()
    return total_rewards,Overall_Delivered_Packets,Overall_Excess_Energy,Overall_Number_of_Contacts_Used,agent_specific_delivered_packets,agent_specific_excess_energy,agent_specific_num_of_contacts,total_initial_data_volume,agent_specific_initial_data_volume


In [101]:
#CGR
def BaselineCGR(env,seed):
    observations,info = env.reset()
    
    all_agents = ["satellite1", "satellite2", "satellite3"]
    
    terminated = {agent: False for agent in all_agents}
    truncated = {agent: False for agent in all_agents}
    total_rewards=0
    while not all([terminated[a] or truncated[a] for a in all_agents]):
        actions = {
            agent:1
            for agent in all_agents
            if not (terminated[agent] or truncated[agent])
        }
        observations, rewards, terminated, truncated, info = env.step(actions)
        for a in all_agents:
            total_rewards+=rewards[a]
    satellite1_info=info['satellite1']
    Overall_Delivered_Packets=satellite1_info['Delivered_Packets']
    Overall_Excess_Energy=satellite1_info['Total_Energy_Expended']
    Overall_Number_of_Contacts_Used=satellite1_info['Total_Number_of_Contacts_Used']
    num_of_collisions=satellite1_info['Number_of_Collisions']
    total_initial_data_volume=satellite1_info['Total_Initial_Satellite_Data_Volume']
    
    agent_specific_delivered_packets=[]
    agent_specific_excess_energy=[]
    agent_specific_num_of_contacts=[]
    agent_specific_initial_data_volume=[]
    for a in all_agents:
        sat_info=info[a]
        agent_specific_delivered_packets.append(sat_info['satellite_delivered_packets'])
        agent_specific_excess_energy.append(sat_info['satellite_energy_expended'])
        agent_specific_num_of_contacts.append(sat_info['satellite_number_of_contacts_used'])
        agent_specific_initial_data_volume.append(sat_info['satellite_initial_data_volume'])
        
    env.close()
    return total_rewards,Overall_Delivered_Packets,Overall_Excess_Energy,Overall_Number_of_Contacts_Used,agent_specific_delivered_packets,agent_specific_excess_energy,agent_specific_num_of_contacts,total_initial_data_volume,agent_specific_initial_data_volume


In [102]:
#Here we do the testing for each algorithm
Number_of_Episodes=100
seeds=[50,51,52,53,54]
all_agents = ["satellite1", "satellite2", "satellite3"]
CGR_reward_vector=np.zeros(len(seeds)*Number_of_Episodes)
CGR_total_delivery_ratio_vector=np.zeros(len(seeds)*Number_of_Episodes)
CGR_total_energy_expenditure_vector=np.zeros(len(seeds)*Number_of_Episodes)
CGR_total_number_of_contacts=np.zeros(len(seeds)*Number_of_Episodes)
CGR_mean_contact_energy_efficiency_vector=np.zeros(len(seeds)*Number_of_Episodes)
CGR_agent_delivery_ratio_vector=np.zeros([len(seeds)*Number_of_Episodes,len(all_agents)])
CGR_agent_excess_energy_vector=np.zeros([len(seeds)*Number_of_Episodes,len(all_agents)])
CGR_agent_num_of_contacts_vector=np.zeros([len(seeds)*Number_of_Episodes,len(all_agents)])

vector_index=0
for h in range(1,Number_of_Episodes):
        for b in seeds:
            total_reward,Overall_Delivered_Packets,Overall_Excess_Energy,Overall_Number_of_Contacts_Used,agent_specific_delivered_packets,agent_specific_excess_energy,agent_specific_num_of_contacts,initial_data_volume,agent_specific_data_volume=BaselineCGR(env,b)
            vector_index+=1
            #Next need to develop output metrics for analysis
            CGR_reward_vector[vector_index]=total_reward
            CGR_total_delivery_ratio_vector[vector_index]=Overall_Delivered_Packets/initial_data_volume
            CGR_mean_contact_energy_efficiency_vector[vector_index]=Overall_Delivered_Packets/Overall_Number_of_Contacts_Used
            #print(agent_specific_delivered_packets)
            for a in range(0,len(all_agents)):
                CGR_agent_delivery_ratio_vector[vector_index,a]=agent_specific_delivered_packets[a]/agent_specific_data_volume[a]
                CGR_agent_excess_energy_vector[vector_index,a]=agent_specific_excess_energy[a]
                CGR_agent_num_of_contacts_vector[vector_index,a]=agent_specific_num_of_contacts[a]


            
Rand_reward_vector=np.zeros(len(seeds)*Number_of_Episodes)
Rand_total_delivery_ratio_vector=np.zeros(len(seeds)*Number_of_Episodes)
Rand_total_energy_expenditure_vector=np.zeros(len(seeds)*Number_of_Episodes)
Rand_total_number_of_contacts=np.zeros(len(seeds)*Number_of_Episodes)
Rand_mean_contact_energy_efficiency_vector=np.zeros(len(seeds)*Number_of_Episodes)
Rand_agent_delivery_ratio_vector=np.zeros([len(seeds)*Number_of_Episodes,len(all_agents)])
Rand_agent_excess_energy_vector=np.zeros([len(seeds)*Number_of_Episodes,len(all_agents)])
Rand_agent_num_of_contacts_vector=np.zeros([len(seeds)*Number_of_Episodes,len(all_agents)])

vector_index=0
for h in range(1,Number_of_Episodes):
        for b in seeds:
            total_reward,Overall_Delivered_Packets,Overall_Excess_Energy,Overall_Number_of_Contacts_Used,agent_specific_delivered_packets,agent_specific_excess_energy,agent_specific_num_of_contacts,initial_data_volume,agent_specific_data_volume=BaselineCGR(env,b)
            vector_index+=1
            #Next need to develop output metrics for analysis
            Rand_reward_vector[vector_index]=total_reward
            Rand_total_delivery_ratio_vector[vector_index]=Overall_Delivered_Packets/initial_data_volume
            Rand_mean_contact_energy_efficiency_vector[vector_index]=Overall_Delivered_Packets/Overall_Number_of_Contacts_Used
            #print(agent_specific_delivered_packets)
            for a in range(0,len(all_agents)):
                Rand_agent_delivery_ratio_vector[vector_index,a]=agent_specific_delivered_packets[a]/agent_specific_data_volume[a]
                Rand_agent_excess_energy_vector[vector_index,a]=agent_specific_excess_energy[a]
                Rand_agent_num_of_contacts_vector[vector_index,a]=agent_specific_num_of_contacts[a]

Sort_reward_vector=np.zeros(len(seeds)*Number_of_Episodes)
Sort_total_delivery_ratio_vector=np.zeros(len(seeds)*Number_of_Episodes)
Sort_total_energy_expenditure_vector=np.zeros(len(seeds)*Number_of_Episodes)
Sort_total_number_of_contacts=np.zeros(len(seeds)*Number_of_Episodes)
Sort_mean_contact_energy_efficiency_vector=np.zeros(len(seeds)*Number_of_Episodes)
Sort_agent_delivery_ratio_vector=np.zeros([len(seeds)*Number_of_Episodes,len(all_agents)])
Sort_agent_excess_energy_vector=np.zeros([len(seeds)*Number_of_Episodes,len(all_agents)])
Sort_agent_num_of_contacts_vector=np.zeros([len(seeds)*Number_of_Episodes,len(all_agents)])
vector_index=0
for h in range(1,Number_of_Episodes):
        for b in seeds:
            total_reward,Overall_Delivered_Packets,Overall_Excess_Energy,Overall_Number_of_Contacts_Used,agent_specific_delivered_packets,agent_specific_excess_energy,agent_specific_num_of_contacts,initial_data_volume,agent_specific_data_volume=AdaptiveSorting(env,b)
            vector_index+=1
            Sort_reward_vector[vector_index]=total_reward
            Sort_total_delivery_ratio_vector[vector_index]=Overall_Delivered_Packets/initial_data_volume
            Sort_mean_contact_energy_efficiency_vector[vector_index]=Overall_Delivered_Packets/Overall_Number_of_Contacts_Used
            #print(agent_specific_delivered_packets)
            for a in range(0,len(all_agents)):
                Sort_agent_delivery_ratio_vector[vector_index,a]=agent_specific_delivered_packets[a]/agent_specific_data_volume[a]
                Sort_agent_excess_energy_vector[vector_index,a]=agent_specific_excess_energy[a]
                Sort_agent_num_of_contacts_vector[vector_index,a]=agent_specific_num_of_contacts[a]


[ 3.8753721e-01  1.0000000e+01  1.0000000e+00  0.0000000e+00
  0.0000000e+00  9.4391555e-01  1.0000000e+01  1.0000000e+00
  1.0000000e+00  1.0000000e+00  3.1056815e-01  1.0000000e+01
  1.0000000e+00  1.0000000e+00  1.0000000e+00  6.8633646e-02
  1.0000000e+01  1.0000000e+00  0.0000000e+00  1.0000000e+00
  8.3543760e-01  1.0000000e+01  0.0000000e+00  1.0000000e+00
  0.0000000e+00  3.6371225e-01  1.0000000e+01  1.0000000e+00
  1.0000000e+00  1.0000000e+00  7.6227373e-01  1.0000000e+01
  1.0000000e+00  1.0000000e+00  0.0000000e+00  5.0582016e-01
  1.0000000e+01  0.0000000e+00  1.0000000e+00  1.0000000e+00
  6.1492431e-01  1.0000000e+01  1.0000000e+00  0.0000000e+00
  0.0000000e+00  9.5153028e-01  1.0000000e+01  1.0000000e+00
  1.0000000e+00  1.0000000e+00  8.5969192e-01  1.0000000e+01
  1.0000000e+00  0.0000000e+00  1.0000000e+00  3.9846075e-01
  1.0000000e+01  1.0000000e+00  1.0000000e+00  0.0000000e+00
  5.6011891e-01  1.0000000e+01  0.0000000e+00  1.0000000e+00
  0.0000000e+00  4.20956

ValueError: all the input array dimensions except for the concatenation axis must match exactly, but along dimension 0, the array at index 0 has size 27 and the array at index 1 has size 30

In [ ]:
import matplotlib.pyplot as plt
#Reward Plot
print(np.mean(Sorting_reward_vector))
plt.boxplot([Rand_reward_vector,CGR_reward_vector, Sorting_reward_vector], tick_labels=['Random','CGR', 'Sorting Algorithm'])

plt.title("Rewards ")
plt.ylabel("rewards")
plt.show()

#Delivery Ratio Plot
plt.boxplot([Rand_delivery_ratio_vector,CGR_delivery_ratio_vector, Sorting_delivery_ratio_vector], tick_labels=['Random','CGR', 'Sorting Algorithm'])

plt.title("Delivery Ratio")
plt.ylabel("delivery ratio")
plt.show()
#Contact Energy Efficiency Plot
plt.boxplot([CGR_mean_contact_energy_efficiency_vector, Sorting_mean_contact_energy_efficiency_vector], tick_labels=['CGR', 'Sorting Algorithm'])

plt.title("Box and Whisker Plot")
plt.ylabel("Values")
plt.show()